In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from loguru import logger
logger.remove()  # keep the slides quiet

import numpy as np
import matplotlib.pyplot as plt

from fl_experiment_setup import ModelSpec, DesignSpec
from fl_experiment_runner import run_experiment
from sim_theorem_partii import DispersionBiasExperiment

OUT_DIR = REPO_ROOT / "nb_outputs"
OUT_DIR.mkdir(exist_ok=True)

# Same canonical diagonal-Gram model as theorem_partii_clean_heavy_tail.ipynb.
model = ModelSpec(
    k_factors=3,
    factor_vols=[0.16, 0.08, 0.06],
    beta_samplers=[
        {"distribution": "normal", "loc": 1.0, "scale": 0.5},   # market-like factor
        {"distribution": "normal", "loc": 0.0, "scale": 1.0},   # zero-mean
        {"distribution": "normal", "loc": 0.0, "scale": 1.0},   # zero-mean
    ],
    idio_vol_sampler={"distribution": "constant", "value": 0.4},
)
design = DesignSpec(
    n_values=[63],
    p_values=[200, 500, 1000, 2000, 5000, 10000, 20000],
    n_reps=300,
    random_seed=20260511,
    sampling="nested",
    nest_time=False,
    # Heavy-tail variant: swap in student-t samplers, e.g.
    # factor_return_sampler={"distribution": "student_t", "df": 3, "loc": 0.0, "scale": 1.0},
    # idio_return_sampler={"distribution": "student_t", "df": 4, "loc": 0.0, "scale": 1.0},
)

df = run_experiment(model, design, DispersionBiasExperiment())

# Plot-ready angles, named to the Eq.(17) decomposition  (paper symbol -> column -> here):
#   measured  sin^2 angle(h_j, b_j)              sin2_j  -> ang_meas
#   out-of-subspace + in-subspace (full RHS)     rhs     -> ang_theory
#   out-of-subspace error  d^2/(n*lambda+d^2)    floor   -> ang_oos
#   lambda_{n,j} = eig of D_hat = C^{1/2}(F'F/n)C^{1/2}  (renormalized; c=[2,1,1])  -> column 'rho'
# to_deg maps sin^2 -> angle in degrees.
def to_deg(v):
    """sin^2 of an angle -> the angle in degrees."""
    return np.degrees(np.arcsin(np.sqrt(np.clip(v, 0.0, 1.0))))

d = df.assign(
    s2_meas=df["sin2_j"], s2_theory=df["rhs"], s2_oos=df["floor"],          # additive frame (sin^2)
    ang_meas=to_deg(df["sin2_j"]), ang_theory=to_deg(df["rhs"]), ang_oos=to_deg(df["floor"]),
)
def summarize(frame, key):
    """Per (key, factor j): component means + SEMs of measured/theory and of the paired
    gap (= measured - theory), over the replicate axis. SE = sd/sqrt(n_reps)."""
    g = frame.assign(gap=frame["s2_meas"] - frame["s2_theory"])
    return g.groupby([key, "j"]).agg(
        s2_meas=("s2_meas", "mean"),       s2_meas_se=("s2_meas", "sem"),
        s2_theory=("s2_theory", "mean"),   s2_theory_se=("s2_theory", "sem"),
        s2_oos=("s2_oos", "mean"),
        ang_meas=("ang_meas", "mean"),     ang_meas_se=("ang_meas", "sem"),
        ang_theory=("ang_theory", "mean"), ang_theory_se=("ang_theory", "sem"),
        ang_oos=("ang_oos", "mean"),
        gap=("gap", "mean"),               gap_se=("gap", "sem"),
    ).reset_index()

avg = summarize(d, "p")
P_VALUES = sorted(d["p"].unique())

NAVY, RED, GRAY = "#1f3864", "#c0392b", "#555555"
FACTOR_COLORS = ["tab:blue", "tab:orange", "tab:green"]
DEG_TICKS = ([0, 30, 60, 90], ["0\u00b0", "30\u00b0", "60\u00b0", "90\u00b0"])
print(f"rows: {len(df)}   p sweep: {P_VALUES}")

SyntaxError: unterminated string literal (detected at line 30) (4142206058.py, line 30)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.3, 4.6), sharey=True)
fig.subplots_adjust(left=0.06, right=0.97, top=0.86, bottom=0.28)

x = np.arange(len(P_VALUES))
for j, ax in zip((1, 2, 3), axes):
    a = avg[avg["j"] == j].set_index("p").loc[P_VALUES]
    oos_part = a["ang_oos"].to_numpy()
    insub_part = a["ang_theory"].to_numpy() - oos_part
    ax.bar(x, oos_part, color="#4878a8", label="out-of-subspace error (estimable, large)")
    ax.bar(x, insub_part, bottom=oos_part, color="#f28e2b",
           label="in-subspace error (latent, small)")
    ax.plot(x, a["ang_meas"].to_numpy(), "o-", color="black",
            label=r"measured $\angle(h, \bar b)$")
    ax.set_xticks(x, [f"{p:,}" for p in P_VALUES], fontsize=8)
    ax.set_ylim(0, 90)
    ax.set_yticks(*DEG_TICKS)
    ax.set_title(f"factor {j}", color=NAVY)
    ax.set_xlabel("p (assets)")
axes[0].set_ylabel("average angle")
axes[0].legend(fontsize=8, loc="upper right")

fig.text(0.5, 0.04,
         "Blue (out-of-subspace error) dominates the angle at every dimension p; orange (in-subspace error"
         ") is a thin sliver.\nBoth pieces are set by the realized factor returns, "
         "not by p \u2014 the note's punchline: the recoverable part of the error is the larger one.",
         fontsize=11, color=GRAY, ha="center")

fig.savefig(OUT_DIR / "slide_partii_check_reveals.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plot styling + decomposition / gap / share helpers shared by the panels below.
N_FIXED = design.n_values[0]   # n held fixed across the p sweep (= 63)
FACTOR_COLORS = ["tab:blue", "tab:orange", "tab:green"]
NAVY, GRAY = "#1f3864", "#555555"
DEG_TICKS  = ([0, 30, 60, 90], ["0°", "30°", "60°", "90°"])
SIN2_TICKS = ([0, 0.25, 0.5, 0.75, 1.0], ["0", "0.25", "0.50", "0.75", "1.0"])

# Eq. (17): sin^2 angle(h, b) = out-of-subspace + in-subspace  (additive in sin^2).
LABEL_MEAS  = r"measured $\angle(h, \bar b)$"
LABEL_OOS   = r"out-of-subspace: $\delta^2/(n\lambda_{n,j}+\delta^2)$"
LABEL_INSUB = "in-subspace"

def decomp_panel(ax, cats, meas, oos, insub, meas_se=None, total_se=None, width=0.7):
    """Stacked bars for the theory decomposition (out-of-subspace + in-subspace),
    with the measured value drawn as a black line-with-markers on top.
    Optional 95% CI caps (yerr = 1.96 * SE) on the measured line and the stack total."""
    x = np.arange(len(cats))
    ax.bar(x, oos,   width, color="#4878a8", label=LABEL_OOS)
    ax.bar(x, insub, width, bottom=oos, color="#f28e2b", label=LABEL_INSUB)
    ax.plot(x, meas, "o-", color="black", label=LABEL_MEAS, zorder=5)
    cap = dict(fmt="none", ecolor="black", elinewidth=0.8, capsize=2, zorder=5)
    if meas_se is not None:
        ax.errorbar(x, meas, yerr=1.96 * meas_se, **cap)
    if total_se is not None:
        ax.errorbar(x, oos + insub, yerr=1.96 * total_se, **cap)
    ax.set_xticks(x, cats, fontsize=8)
    return x

def gap_panel(ax, cats, gap, gap_se, color):
    """Gap = mean(measured - theory) in sin^2, with 95% CI and a zero reference.
    It is the per-replicate (measured - Eq.(17) prediction) difference averaged over
    the R replicates: 0 means theory matches; >0 means the measured misalignment
    exceeds the prediction; <0 the reverse."""
    x = np.arange(len(cats))
    ax.axhline(0, color="0.6", lw=0.8, ls="--")
    ax.errorbar(x, gap, yerr=1.96 * gap_se, fmt="o-", color=color, ms=4, lw=1.2, capsize=2)
    ax.set_xticks(x, cats, fontsize=8)
    return x

def frac_panel(ax, cats, frac, color):
    """Bottom diagnostic (dimensionless): in-subspace share = in-sub / (oos + in-sub)
    of the predicted total -- how much of the predicted misalignment is the
    recoverable in-subspace piece vs the out-of-subspace floor.
    ("share" is a placeholder name -- the term for this is TBD.)"""
    x = np.arange(len(cats))
    ax.bar(x, frac, 0.7, color=color, alpha=0.55, label="in-subspace share")
    ax.set_xticks(x, cats, fontsize=8)
    return x

In [ ]:
# Growing p, fixed n: Eq. (17) decomposition + diagnostics (same 9-panel format as the growing-n cell below).
# Rows: sin^2 (oos + in-subspace, measured line) | gap = measured - theory | in-subspace share (bars). Caps/strips = 95% CI (R = 300).
fig, axes = plt.subplots(3, 3, figsize=(13.3, 9.6), sharex="col", sharey="row",
                         gridspec_kw={"height_ratios": [1, 0.7, 0.7]})
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.17, hspace=0.13, wspace=0.08)
cats = [f"{p:,}" for p in P_VALUES]
for j in (1, 2, 3):
    a = avg[avg["j"] == j].set_index("p").loc[P_VALUES]
    decomp_panel(axes[0, j-1], cats, a["s2_meas"].to_numpy(), a["s2_oos"].to_numpy(),
                 (a["s2_theory"] - a["s2_oos"]).to_numpy(),
                 meas_se=a["s2_meas_se"].to_numpy(), total_se=a["s2_theory_se"].to_numpy())
    axes[0, j-1].set_title(f"factor {j}", color=NAVY)
    gap_panel(axes[1, j-1], cats, a["gap"].to_numpy(), a["gap_se"].to_numpy(), FACTOR_COLORS[j-1])
    insub = (a["s2_theory"] - a["s2_oos"]).to_numpy()
    frac_panel(axes[2, j-1], cats, insub / a["s2_theory"].to_numpy(), FACTOR_COLORS[j-1])
    axes[2, j-1].set_xlabel("p (assets)")
axes[0, 0].set_ylim(0, 1); axes[0, 0].set_yticks(*SIN2_TICKS); axes[0, 0].set_ylabel(r"average $\sin^2$")
axes[1, 0].set_ylabel("gap = mean(meas − theory)\n[sin², paired]")
axes[2, 0].set_ylabel("in-subspace share")
axes[0, 0].legend(fontsize=8, loc="upper right")
for ax in axes.flat:
    ax.label_outer()
fig.suptitle(f"Growing p, fixed n = {design.n_values[0]}", color=NAVY, y=0.985)
fig.text(0.5, 0.025,
         "Top — sin²∠(h, b̄) = out-of-subspace + in-subspace (additive); black line = measured; caps = 95% CI (R = 300, SE = sd/√R).   "
         "Middle — gap = mean(measured − theory) sin² (paired): 0 ⇒ theory matches, >0 ⇒ measured exceeds prediction.   "
         "Bottom — in-subspace share = in-sub/(oos+in-sub) of the predicted total.",
         ha="center", va="top", fontsize=8, color=GRAY, wrap=True)
plt.show()

In [ ]:
# Fixed p = 3000, growing n: sin^2 decomposition | gap | in-subspace share.
P_FIXED = 3000
design_n = DesignSpec(
    n_values=[20, 30, 45, 60, 90, 120, 180, 250],
    p_values=[P_FIXED],
    n_reps=300,
    random_seed=20260511,
    sampling="nested",
)
df_n = run_experiment(model, design_n, DispersionBiasExperiment())
d_n = df_n.assign(
    s2_meas=df_n["sin2_j"], s2_theory=df_n["rhs"], s2_oos=df_n["floor"],
    ang_meas=to_deg(df_n["sin2_j"]), ang_theory=to_deg(df_n["rhs"]), ang_oos=to_deg(df_n["floor"]),
)
avg_n = summarize(d_n, "n")
N_VALUES = sorted(d_n["n"].unique())
fig, axes = plt.subplots(3, 3, figsize=(13.3, 9.6), sharex="col", sharey="row",
                         gridspec_kw={"height_ratios": [1, 0.7, 0.7]})
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.17, hspace=0.13, wspace=0.08)
cats = [str(n) for n in N_VALUES]
for j in (1, 2, 3):
    a = avg_n[avg_n["j"] == j].set_index("n").loc[N_VALUES]
    decomp_panel(axes[0, j-1], cats, a["s2_meas"].to_numpy(), a["s2_oos"].to_numpy(),
                 (a["s2_theory"] - a["s2_oos"]).to_numpy(),
                 meas_se=a["s2_meas_se"].to_numpy(), total_se=a["s2_theory_se"].to_numpy())
    axes[0, j-1].set_title(f"factor {j}", color=NAVY)
    gap_panel(axes[1, j-1], cats, a["gap"].to_numpy(), a["gap_se"].to_numpy(), FACTOR_COLORS[j-1])
    insub = (a["s2_theory"] - a["s2_oos"]).to_numpy()
    frac_panel(axes[2, j-1], cats, insub / a["s2_theory"].to_numpy(), FACTOR_COLORS[j-1])
    axes[2, j-1].set_xlabel("n (periods)")
axes[0, 0].set_ylim(0, 1); axes[0, 0].set_yticks(*SIN2_TICKS); axes[0, 0].set_ylabel(r"average $\sin^2$")
axes[1, 0].set_ylabel("gap = mean(meas − theory)\n[sin², paired]")
axes[2, 0].set_ylabel("in-subspace share")
axes[0, 0].legend(fontsize=8, loc="upper right")
for ax in axes.flat:
    ax.label_outer()
fig.suptitle(f"Fixed p = {P_FIXED:,}, growing n", color=NAVY, y=0.985)
fig.text(0.5, 0.025,
         "Top — sin²∠(h, b̄) = out-of-subspace + in-subspace (additive); black line = measured; caps = 95% CI (R = 300, SE = sd/√R).   "
         "Middle — gap = mean(measured − theory) sin² (paired): 0 ⇒ theory matches, >0 ⇒ measured exceeds prediction.   "
         "Bottom — in-subspace share = in-sub/(oos+in-sub) of the predicted total.",
         ha="center", va="top", fontsize=8, color=GRAY, wrap=True)
plt.show()

In [ ]:
# Same fixed-p = 3000 sweep with nest_time=True (n axis nested): smoother n-curve.
P_FIXED = 3000
design_nt = DesignSpec(
    n_values=[20, 30, 45, 60, 90, 120, 180, 250],
    p_values=[P_FIXED],
    n_reps=300,
    random_seed=20260511,
    sampling="nested",
    nest_time=True,
)
df_nt = run_experiment(model, design_nt, DispersionBiasExperiment())
d_nt = df_nt.assign(
    s2_meas=df_nt["sin2_j"], s2_theory=df_nt["rhs"], s2_oos=df_nt["floor"],
    ang_meas=to_deg(df_nt["sin2_j"]), ang_theory=to_deg(df_nt["rhs"]), ang_oos=to_deg(df_nt["floor"]),
)
avg_nt = summarize(d_nt, "n")
N_VALUES_NT = sorted(d_nt["n"].unique())
fig, axes = plt.subplots(3, 3, figsize=(13.3, 9.6), sharex="col", sharey="row",
                         gridspec_kw={"height_ratios": [1, 0.7, 0.7]})
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.17, hspace=0.13, wspace=0.08)
cats = [str(n) for n in N_VALUES_NT]
for j in (1, 2, 3):
    a = avg_nt[avg_nt["j"] == j].set_index("n").loc[N_VALUES_NT]
    decomp_panel(axes[0, j-1], cats, a["s2_meas"].to_numpy(), a["s2_oos"].to_numpy(),
                 (a["s2_theory"] - a["s2_oos"]).to_numpy(),
                 meas_se=a["s2_meas_se"].to_numpy(), total_se=a["s2_theory_se"].to_numpy())
    axes[0, j-1].set_title(f"factor {j}", color=NAVY)
    gap_panel(axes[1, j-1], cats, a["gap"].to_numpy(), a["gap_se"].to_numpy(), FACTOR_COLORS[j-1])
    insub = (a["s2_theory"] - a["s2_oos"]).to_numpy()
    frac_panel(axes[2, j-1], cats, insub / a["s2_theory"].to_numpy(), FACTOR_COLORS[j-1])
    axes[2, j-1].set_xlabel("n (periods)")
axes[0, 0].set_ylim(0, 1); axes[0, 0].set_yticks(*SIN2_TICKS); axes[0, 0].set_ylabel(r"average $\sin^2$")
axes[1, 0].set_ylabel("gap = mean(meas − theory)\n[sin², paired]")
axes[2, 0].set_ylabel("in-subspace share")
axes[0, 0].legend(fontsize=8, loc="upper right")
for ax in axes.flat:
    ax.label_outer()
fig.suptitle(f"Fixed p = {P_FIXED:,}, growing n — nest_time=True (n axis nested)", color=NAVY, y=0.985)
fig.text(0.5, 0.025,
         "Top — sin²∠(h, b̄) = out-of-subspace + in-subspace (additive); black line = measured; caps = 95% CI (R = 300, SE = sd/√R).   "
         "Middle — gap = mean(measured − theory) sin² (paired): 0 ⇒ theory matches, >0 ⇒ measured exceeds prediction.   "
         "Bottom — in-subspace share = in-sub/(oos+in-sub) of the predicted total.",
         ha="center", va="top", fontsize=8, color=GRAY, wrap=True)
plt.show()

In [ ]:
# Lisa's note: "For growing n, we show (intuitively) that the total error and the
# fraction of error due to the latent term tend to decrease as n increases.
# I wonder if we should add a figure featuring the fraction." Left panel is the
# total theoretical error (floor + rotation); right panel is the rotation term's
# share of that total. Both vs n, using the nest_time=True sweep (avg_nt) for a
# smoother curve.
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
x = np.arange(len(N_VALUES_NT))
cats = [str(n) for n in N_VALUES_NT]
for j, color in zip((1, 2, 3), FACTOR_COLORS):
    a = avg_nt[avg_nt["j"] == j].set_index("n").loc[N_VALUES_NT]
    total = a["s2_theory"].to_numpy()
    frac = ((a["s2_theory"] - a["s2_oos"]) / a["s2_theory"]).to_numpy()
    axes[0].plot(x, total, "o-", color=color, label=f"factor {j}")
    axes[1].plot(x, frac,  "o-", color=color, label=f"factor {j}")
axes[0].set_title("Total error (floor + rotation)", color=NAVY)
axes[0].set_ylabel(r"$\sin^2$")
axes[1].set_title("In-subspace (rotation) share of total error", color=NAVY)
axes[1].set_ylabel("fraction of total error")
for ax in axes:
    ax.set_xticks(x, cats, fontsize=8)
    ax.set_xlabel("n (periods)")
    ax.set_ylim(0, 1)
axes[0].legend(fontsize=8)
fig.suptitle(f"Fixed p = {P_FIXED:,}: total error and in-subspace share, growing n",
              color=NAVY, y=1.0)
plt.show()

In [ ]:
# Diagnostics tables (growing n): per-factor in-subspace share and relative gap.
#   in-subspace share = in-sub / (oos + in-sub) of the predicted total   ("error fraction")
#   relative gap       = (measured - theory) / measured                  ("relative error")
# (Names are placeholders -- the term for these is TBD.) Shown for both n-sweeps:
# independent sampling (avg_n) and the nest_time sweep (avg_nt).
import pandas as pd

def diagnostics_table(frame, key):
    t = frame.assign(
        in_sub_share=(frame["s2_theory"] - frame["s2_oos"]) / frame["s2_theory"],
        rel_gap=frame["gap"] / frame["s2_meas"],
    )
    cols = {}
    for j in sorted(t["j"].unique()):
        fj = t[t["j"] == j].set_index(key)
        cols[(f"factor {j}", "in-sub share")] = fj["in_sub_share"]
        cols[(f"factor {j}", "rel gap")] = fj["rel_gap"]
    out = pd.DataFrame(cols).sort_index()
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    out.index.name = key
    return out

print("Growing n, independent sampling -- in-subspace share & relative gap by factor")
display(diagnostics_table(avg_n, "n").round(4))
print("\nGrowing n, nest_time sampling -- in-subspace share & relative gap by factor")
display(diagnostics_table(avg_nt, "n").round(4))

## Ethan's Grassmann-distance boxplot (sample-target vs sample-truth)

Lifted from `Grassmann_simulation_Ethan.py` on the `multi-d-and-n-boxplot` branch. Self-contained (own simulation, scipy/seaborn) and independent of the `factor_lab` engine above. **Heavy:** sweeps p up to 10000 over 100 trials x 100 perturbations x 5 radii x 2 n — expect a few minutes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from typing import Tuple
from scipy.linalg import expm, subspace_angles, qr, det
from scipy.sparse.linalg import eigsh
sqrt = np.sqrt

p_values = [100, 200, 500, 1000, 2000, 5000, 10000]
k = 2
ns = [63, 126]
radii = [0.1, 0.3, 0.5, 0.7, 0.9]
n_trials = 100
factor_variances = np.array([.16**2, 0.05**2])
idio_variance = .25
n_perturbations = 100
seed = 42
#k = 3 factor model, 16% 5% and 5% volatility 0.5 specific risk



if len(factor_variances) > k:
    factor_variances = factor_variances[:k]

def orthonormalize(B: np.ndarray) -> np.ndarray:
    Q, _ = qr(B.T, mode='economic')
    return Q
#QR decomp 

def compute_grassmannian_distance(B_true, B_estimated):
    Q_true = orthonormalize(B_true)
    Q_estimated = orthonormalize(B_estimated)
    angles = subspace_angles(Q_true, Q_estimated)
    distance = float(np.linalg.norm(angles))
    return distance, angles
#takes two frames and then finds the Grassmann Distance

def generate_loadings(p, k, rng):
    B = np.zeros((k, p))
    B[0, :] = rng.normal(1.0, 1.0, p)
    for i in range(1, k):
        B[i, :] = rng.normal(0.0, 1.0, p)
    return B
#creates true factor matrix B

def simulate_returns(B, factor_variances, idio_variance, T, rng):
    k, p = B.shape
    f = rng.normal(size=(T, k)) * np.sqrt(factor_variances)
    e = rng.normal(size=(T, p)) * np.sqrt(idio_variance)
    return f @ B + e
#generates observations from the model 

def estimate_factors(returns, k):
    X = returns - returns.mean(axis=0)
    _, s, Vt = np.linalg.svd(X, full_matrices=False)
    return Vt[:k, :].T
#SVD on the returns and returns top k vectors 

def construct_epsilon_distance_perturbation(t, B, direction_rng):
    p, k = B.shape
    if k > p:
        B = B.T
        p, k = B.shape

    Z = direction_rng.standard_normal(B.shape)
    Delta = Z - B @ ((B.T @ Z + Z.T @ B) / 2)
    Delta = Delta / np.linalg.norm(Delta, 'fro')

    A = B.T @ Delta
    A = (A - A.T) / 2

    Delta_perp = Delta - B @ A
    Q_perp, R = np.linalg.qr(Delta_perp, mode='reduced')

    M = np.block([[A, -R.T],
                  [R, np.zeros((k, k))]])

    E = expm(t * M)
    M_t = E[:k, :k]
    N_t = E[k:, :k]

    return (B @ M_t + Q_perp @ N_t).T
#generaetes perturbations a geodesic away from the truth

def thm31_check(Q_est, Q_true, perturbation):
    k = Q_est.shape[1]
    if k > 1:
        h, b, z = Q_est, Q_true, perturbation
        lhs = np.abs(det(h.T @ z))
        rhs = np.abs(det(h.T @ b)) * np.abs(det(b.T @ z))
        return lhs, rhs
    h = Q_est[:, 0]
    b = Q_true[:, 0]
    z = perturbation
    lhs = abs(np.dot(h, z))
    rhs = abs(np.dot(h, b)) * abs(np.dot(b, z))
    return lhs, rhs
#Checks theorem 3.1, inner product / determinant method

#Pre-generate B and returns once at p_max
p_max = max(p_values)
rng_B = np.random.default_rng(seed)
B_max = generate_loadings(p_max, k, rng_B)

returns_collection = []
rng_sim = np.random.default_rng(p_max * seed)
for t in range(n_trials):
    returns_collection.append(
        simulate_returns(B_max, factor_variances, idio_variance, max(ns), rng_sim)
    )

#Main loop
records = []
thm31_records = []
perturbation_check = max(radii)

for p in p_values:
    B = B_max[:, :p]
    Q_true = orthonormalize(B)  #(p, k)

    for radius in radii:
        direction_rng = np.random.default_rng(seed + 2)
        perturbed_frames = []
        for _ in range(n_perturbations):
            pf = construct_epsilon_distance_perturbation(radius, Q_true, direction_rng)
            perturbed_frames.append(orthonormalize(pf))

        for n in ns:
            for t in range(n_trials):
                returns_used = returns_collection[t][:n, :p]
                Q_est = estimate_factors(returns_used, k)

                #sample-truth
                d_truth, _ = compute_grassmannian_distance(Q_true.T, Q_est.T)
                records.append({
                    'dimension': k,
                    'p': p,
                    'n': n,
                    'radius': radius,
                    'distance_type': 'sample-truth',
                    'distance': d_truth,
                    'metric': 'grassmann',
                })

                #sample-target
                for Q_perturb in perturbed_frames[:10]:
                    d_target, _ = compute_grassmannian_distance(Q_est.T, Q_perturb.T)
                    records.append({
                        'dimension': k,
                        'p': p,
                        'n': n,
                        'radius': radius,
                        'distance_type': 'sample-target',
                        'distance': d_target,
                        'metric': 'grassmann',
                    })

                #Theorem 3.1 check at largest radius only
                if radius == perturbation_check:
                    z = perturbed_frames[0]
                    lhs, rhs = thm31_check(Q_est, Q_true, z)
                    thm31_records.append({'p': p, 'n': n, 'lhs': lhs, 'rhs': rhs})

    print(f'p={p} done')

#Build DataFrames
long_df = pd.DataFrame(records)
long_df['radius_label'] = long_df['radius'].map(lambda x: f'r={x:.1f}')
long_df['n_label'] = long_df['n'].map(lambda x: f'n={x}')
thm31_df = pd.DataFrame(thm31_records)

print(long_df.head())
print(f'Total records: {len(long_df)}')

In [ ]:
#Darwin plots 
col_order = [f'r={r:.1f}' for r in sorted(long_df['radius'].unique())]
row_order = [f'n={n}' for n in sorted(long_df['n'].unique())]
radius_map = {f'r={r:.1f}': r for r in sorted(long_df['radius'].unique())}

plot_df = long_df[long_df['distance_type'].isin(['sample-truth', 'sample-target'])].copy()

sns.set_theme(style='whitegrid', context='paper')
g = sns.catplot(
    data=plot_df,
    kind='box',
    x='p',
    y='distance',
    hue='distance_type',
    col='radius_label',
    row='n_label',
    col_order=col_order,
    row_order=row_order,
    hue_order=['sample-target', 'sample-truth'],
    sharey=True,
    height=3.0,
    aspect=1.1,
    linewidth=0.8,
    showfliers=False,
)
for axes_row in g.axes:
    for label, ax in zip(col_order, axes_row):
        ax.axhline(radius_map[label], ls='--', lw=1.2, color='black', alpha=0.7)
        ax.set_xlabel('Ambient dimension (p)')
        ax.set_ylabel('Distance')

g.set_titles(row_template='{row_name}', col_template='{col_name}')
g.fig.suptitle(f'Grassmann Distance: Sample vs Target and Truth (k={k})', fontsize=14)
g.fig.subplots_adjust(top=0.90)
if g._legend:
    g._legend.set_title('')
# Save a copy-paste-ready PNG (opaque white bg) for Gmail, and show it inline.
out_png = OUT_DIR / f"grassmann_panel_k{k}.png"
g.savefig(out_png, dpi=150, bbox_inches="tight", facecolor="white")
print(f"saved -> {out_png}")
print("To put it in Gmail: drag that PNG into the compose window, "
      "or right-click the image below and choose Copy Image / Save Image.")
plt.show()